---

# University of Liverpool

## COMP534 - Applied AI

---

This notebook is associated with Assignment 1. Use it to complete the assignment by following the instructions provided in each section, which includes a text cell describing the requirements. For additional details, see the Canvas.

Use this first cell to import the necessary libraries.

In [ ]:
# IMPORTS 

import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Model selection & evaluation 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Pipelines & preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm

# Metrics 
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)



# 1. **Data Management**


In this part, you need to:

1.   analyse and prepare the data. Use plots, graphs, and tables (such as histogram, box plots, scatterplots, etc.) to better analyse the dataset and identify issues or potential improvements in the data, including (but not limited to) unnecessary feature/variable which can be dropped/removed, standardization, encoding, etc;
2.	define an appropriate experimental protocol (such as cross-validation, k-fold, etc).


In [ ]:
# Write your proposed solution code here. Create more code cells if you find it necessary

# Load the prepared dataset 
df = pd.read_csv("dataset_prepared.csv")

print("First rows of the dataset:")
display(df.head())

In [ ]:
# TARGET DISTRIBUTION (DIAGNOSIS) 
plt.figure(figsize=(4,3))
df["diagnosis"].value_counts().plot(kind="bar")
plt.title("Diagnosis class distribution")
plt.ylabel("Count")
plt.show()





In [ ]:
# DROP NON-INFORMATIVE COLUMNS 
df = df.drop(columns=["id", "country"])





In [ ]:

#SPLIT INTO FEATURES (X) AND TARGET LABEL (y)

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Features X shape:", X.shape)
print("Target y shape:", y.shape)

print("\nExample rows from X:")
display(X.head())
print("\nExample values from y:")
display(y.head())



In [ ]:
# IDENTIFY NUMERIC AND CATEGORICAL FEATURES 

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

X_clean = X.copy()



In [ ]:
# Numeric → median
for col in num_cols:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

# Categorical → most frequent value
cat_cols = [c for c in cat_cols if c in X_clean.columns] 
for col in cat_cols:
    mode_vals = X_clean[col].mode(dropna=True)
    if len(mode_vals) > 0:
        X_clean[col] = X_clean[col].fillna(mode_vals.iloc[0])
    else:
        # if the whole column is missing, fill with a placeholder
        X_clean[col] = X_clean[col].fillna("Unknown")


# ---- One-hot encode categorical variables ----
# Converts categories into numeric columns (0/1)

X_clean = pd.get_dummies(X_clean, columns=cat_cols, drop_first=False)

print("Shape after preprocessing:", X_clean.shape)
display(X_clean.head())

In [ ]:
# TRAIN / TEST SPLIT (STRATIFIED)
# Use X_clean (after missing-value handling + one-hot encoding),
# and keep the test set unseen until final evaluation.

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)

print("\nTrain class counts:\n", y_train.value_counts())
print("\nTest class counts:\n", y_test.value_counts())

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)




---

# 2. **Model Training**

Here, you need to:

1.	select and compare at least three machine learning models (seen/discussed during the lectures) appropriate for your modelling;
2.	if there are hyperparameters in a selected algorithm, define a hyperparameter search protocol (you can define your own), and tune them.


In [ ]:
# k-NEAREST NEIGHBOURS (kNN) - Lab-style manual CV tuning
# We try different values of k and weights, evaluate with k-fold CV, then select the best.
# Note: numeric features are standardised after the train/test split (fit on training only).

k_values = [3, 5, 7]
weight_values = ["uniform", "distance"]

knn_rows = []  # store results for summary table
best_knn_score = -1
best_knn_params = None
best_knn_model = None

for k in k_values:
    for w in weight_values:
        model = KNeighborsClassifier(n_neighbors=k, weights=w)

        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
        mean_score = float(np.mean(scores))
        std_score = float(np.std(scores))

        knn_rows.append({
            "model": "kNN",
            "mean_f1_macro": mean_score,
            "std_f1_macro": std_score,
            "n_neighbors": k,
            "weights": w
        })

        print(f"kNN k={k}, weights={w} -> mean F1_macro={mean_score:.6f}, std={std_score:.6f}")

        if mean_score > best_knn_score:
            best_knn_score = mean_score
            best_knn_params = {"n_neighbors": k, "weights": w}
            best_knn_model = model

print("\nBest kNN params:", best_knn_params)
print("Best kNN CV F1_macro:", best_knn_score)

In [ ]:
# DECISION TREE - Lab-style manual CV tuning
# We tune max_depth and min_samples_split to control complexity (lecture: avoid overfitting).

depth_values = [3, 5, 10, None]
min_split_values = [2, 5, 10]

dt_rows = []  # store results for summary table
best_dt_score = -1
best_dt_params = None
best_dt_model = None

for d in depth_values:
    for m in min_split_values:
        model = DecisionTreeClassifier(random_state=42, max_depth=d, min_samples_split=m)

        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
        mean_score = float(np.mean(scores))
        std_score = float(np.std(scores))

        dt_rows.append({
            "model": "Decision Tree",
            "mean_f1_macro": mean_score,
            "std_f1_macro": std_score,
            "max_depth": d,
            "min_samples_split": m
        })

        print(f"DT depth={d}, min_split={m} -> mean F1_macro={mean_score:.6f}, std={std_score:.6f}")

        if mean_score > best_dt_score:
            best_dt_score = mean_score
            best_dt_params = {"max_depth": d, "min_samples_split": m}
            best_dt_model = model

print("\nDecision Tree best params:", best_dt_params)
print("Decision Tree best CV F1_macro:", best_dt_score)

In [ ]:
# SUPPORT VECTOR MACHINE (SVM) - Lab-style manual CV tuning
# We tune C and kernel as discussed in the SVM lecture.
# Note: numeric features were already standardised during preprocessing (before one-hot encoding).

C_values = [0.1, 1, 10]
kernels = ["linear", "rbf"]

svm_rows = []  # store results for summary table
best_svm_score = -1
best_svm_params = None
best_svm_model = None

for C in C_values:
    for k in kernels:
        model = svm.SVC(C=C, kernel=k)

        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
        mean_score = float(np.mean(scores))
        std_score = float(np.std(scores))

        svm_rows.append({
            "model": "SVM",
            "mean_f1_macro": mean_score,
            "std_f1_macro": std_score,
            "C": C,
            "kernel": k
        })

        print(f"SVM C={C}, kernel={k} -> mean F1_macro={mean_score:.6f}, std={std_score:.6f}")

        if mean_score > best_svm_score:
            best_svm_score = mean_score
            best_svm_params = {"C": C, "kernel": k}
            best_svm_model = model

print("\nSVM best params:", best_svm_params)
print("SVM best CV F1_macro:", best_svm_score)

In [ ]:
# CROSS-VALIDATION SUMMARY TABLE (Lab-style)
# Combine results from our manual CV loops into one table for comparison.

rows = []
rows.extend(knn_rows)
rows.extend(dt_rows)
rows.extend(svm_rows)

cv_summary = pd.DataFrame(rows)
print("Cross-validation summary (macro F1):")
display(cv_summary.sort_values(["mean_f1_macro"], ascending=False))




---

# 3. **Evaluate models**

Here, you need to:

1.	test the model (the best one you obtained from the above stage) on the appropriate set.


In [ ]:
# 3. EVALUATE MODELS ON THE TEST SET
# Choose the model with the highest CV macro-F1, fit on training data, evaluate on unseen test data.

candidates = [
    ("kNN", best_knn_model, best_knn_score, best_knn_params),
    ("Decision Tree", best_dt_model, best_dt_score, best_dt_params),
    ("SVM", best_svm_model, best_svm_score, best_svm_params),
]

best_name, final_model, best_cv_score, best_params = max(candidates, key=lambda x: x[2])

print("Chosen final model:", best_name)
print("Best CV macro F1:", best_cv_score)
print("Best params:", best_params)

# Fit on training set only
final_model.fit(X_train, y_train)

# Predict on test set
y_pred = final_model.predict(X_test)

# Compute and print metrics
print("\n=== Test set performance ===")
print("Accuracy:         ", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Macro F1:         ", f1_score(y_test, y_pred, average="macro"))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

# Confusion matrix as a heatmap
labels = ["CI", "Depression", "Both"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.show()